# Vesuvius Fragment Visualizer

Explore competition training fragments from `data/train` (65-layer Z-stacks with manual ink labels).

**Sections:** label overview · interactive layer browser · side-view projections · depth statistics · patch sampling

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tifffile as tiff
import ipywidgets as widgets
from IPython.display import display

DATA_DIR = Path("../data/train")
FRAGMENTS = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

def load_layer(frag_dir, z):
    return tiff.imread(str(frag_dir / "surface_volume" / f"{z:02d}.tif")).astype(np.float32)

def normalize(arr, mask=None):
    src = arr[mask] if mask is not None else arr.ravel()
    p1, p99 = np.percentile(src, [1, 99])
    return np.clip((arr - p1) / (p99 - p1 + 1e-8), 0, 1)

frag_sel = widgets.Dropdown(options=FRAGMENTS, description="Fragment:")
print(f"Found {len(FRAGMENTS)} fragments: {FRAGMENTS}")
display(frag_sel)

## 1. Load Data

Re-run this cell after changing the dropdown to switch fragments.

In [ ]:
FRAG_ID  = frag_sel.value
frag_dir = DATA_DIR / FRAG_ID
tifs      = sorted((frag_dir / "surface_volume").glob("*.tif"))
num_layers = len(tifs)

mask       = np.array(Image.open(str(frag_dir / "mask.png")).convert("L")) > 128
labels_raw = np.array(Image.open(str(frag_dir / "inklabels.png")).convert("L"))
labels_bin = labels_raw > 128

print(f"Fragment    : {FRAG_ID}")
print(f"Layers      : {num_layers}")
print(f"Shape       : {mask.shape}")
print(f"Total pixels: {mask.size:,}")
print(f"Valid pixels: {mask.sum():,}  ({mask.sum()/mask.size*100:.1f}% of frame)")
print(f"Ink pixels  : {labels_bin.sum():,}  ({labels_bin.sum()/mask.sum()*100:.1f}% of valid region)")

## 2. Label Overview

Show the ground-truth ink annotation alongside the papyrus mask.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(mask, cmap="gray")
axes[0].set_title("Papyrus mask (valid region)"); axes[0].axis("off")

axes[1].imshow(labels_bin, cmap="gray")
axes[1].set_title("Ink labels — ground truth"); axes[1].axis("off")

# Red ink on gray mask background
rgb = np.stack([mask.astype(float) * 0.35] * 3, axis=-1)
rgb[:, :, 0] = np.clip(rgb[:, :, 0] + labels_bin.astype(float) * 0.9, 0, 1)
rgb[:, :, 1] = np.clip(rgb[:, :, 1] - labels_bin.astype(float) * 0.1, 0, 1)
rgb[:, :, 2] = np.clip(rgb[:, :, 2] - labels_bin.astype(float) * 0.1, 0, 1)
axes[2].imshow(np.clip(rgb, 0, 1))
axes[2].set_title("Mask + ink labels (red)"); axes[2].axis("off")

plt.suptitle(f"{FRAG_ID} — label overview", fontsize=13)
plt.tight_layout(); plt.show()

## 3. Interactive Layer Browser

Scrub through all Z-layers. The overlay shows where ink is predicted to be.

In [ ]:
@widgets.interact(z=widgets.IntSlider(min=0, max=num_layers-1, value=num_layers//2, description="Layer"))
def view_layer(z=num_layers//2):
    sl   = load_layer(frag_dir, z)
    disp = normalize(sl, mask)

    overlay = np.stack([disp, disp, disp], axis=-1)
    lbl = labels_bin.astype(float)
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] + lbl * 0.55, 0, 1)
    overlay[:, :, 1] = np.clip(overlay[:, :, 1] - lbl * 0.25, 0, 1)
    overlay[:, :, 2] = np.clip(overlay[:, :, 2] - lbl * 0.25, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{FRAG_ID} — Layer z={z:02d}", fontsize=12)

    axes[0].imshow(disp, cmap="gray")         ; axes[0].set_title("CT slice")                   ; axes[0].axis("off")
    axes[1].imshow(np.clip(overlay, 0, 1))    ; axes[1].set_title("CT + ink label (red=ink)")   ; axes[1].axis("off")
    axes[2].hist(sl[mask].ravel(), bins=200, color="steelblue", alpha=0.8)
    axes[2].set_title("Intensity histogram (masked)"); axes[2].set_xlabel("Raw pixel value")

    plt.tight_layout(); plt.show()

## 4. Side-View Projections (XZ / YZ)

Slice vertically through the Z-stack to reveal papyrus thickness.

In [ ]:
mid_y = np.where(mask.any(axis=1))[0][mask.any(axis=1).sum() // 2]
mid_x = np.where(mask.any(axis=0))[0][mask.any(axis=0).sum() // 2]

print(f"Building XZ cross-section at row y={mid_y} and YZ at col x={mid_x}...")
xz = np.stack([load_layer(frag_dir, z)[mid_y, :] for z in range(num_layers)], axis=0)
yz = np.stack([load_layer(frag_dir, z)[:, mid_x] for z in range(num_layers)], axis=0)

fig, axes = plt.subplots(2, 1, figsize=(16, 7))
axes[0].imshow(normalize(xz), cmap="gray", aspect="auto")
axes[0].set_title(f"XZ cross-section (row y={mid_y}) — bright band = papyrus surface")
axes[0].set_xlabel("X position"); axes[0].set_ylabel("Z layer")

axes[1].imshow(normalize(yz), cmap="gray", aspect="auto")
axes[1].set_title(f"YZ cross-section (col x={mid_x})")
axes[1].set_xlabel("Y position"); axes[1].set_ylabel("Z layer")

plt.suptitle(f"{FRAG_ID} — side-view projections", fontsize=12)
plt.tight_layout(); plt.show()

## 5. Depth Statistics

How does CT density vary across the 65 Z-layers? Ink (carbon black) would appear as local density changes near the surface.

In [ ]:
print("Computing per-layer statistics...")
means, stds, mins, maxs = [], [], [], []
for z in range(num_layers):
    sl = load_layer(frag_dir, z)
    v  = sl[mask]
    means.append(v.mean()); stds.append(v.std())
    mins.append(v.min());   maxs.append(v.max())
means, stds, mins, maxs = map(np.array, [means, stds, mins, maxs])
zs = np.arange(num_layers)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(zs, means, lw=1.5, label="mean")
axes[0].fill_between(zs, means - stds, means + stds, alpha=0.2, label="±1 std")
axes[0].set_title("Mean ± std per layer"); axes[0].set_xlabel("Z layer"); axes[0].legend()
axes[0].grid(True, ls="--", alpha=0.4)

axes[1].fill_between(zs, mins, maxs, alpha=0.25, color="green", label="min–max range")
axes[1].plot(zs, mins, color="green", lw=1); axes[1].plot(zs, maxs, color="green", lw=1)
axes[1].set_title("Min / Max per layer"); axes[1].set_xlabel("Z layer"); axes[1].legend()
axes[1].grid(True, ls="--", alpha=0.4)

axes[2].hist(means, bins=20, color="steelblue", edgecolor="white")
axes[2].set_title("Distribution of layer means"); axes[2].set_xlabel("Mean intensity")

plt.suptitle(f"{FRAG_ID} — depth statistics", fontsize=12)
plt.tight_layout(); plt.show()

## 6. Patch Sampling

Random 224×224 crops from the valid mask region at the surface layer. Red tint = ground-truth ink.

In [ ]:
N_PATCHES = 16
PATCH     = 224
Z_MID     = num_layers // 2
np.random.seed(42)

sl   = load_layer(frag_dir, Z_MID)
H, W = sl.shape
valid_ys, valid_xs = np.where(
    mask[PATCH//2 : H - PATCH//2, PATCH//2 : W - PATCH//2]
)
chosen = np.random.choice(len(valid_ys), N_PATCHES, replace=False)

fig, axes = plt.subplots(4, 4, figsize=(12, 12))
fig.suptitle(f"{FRAG_ID} — {N_PATCHES} random {PATCH}×{PATCH} patches  (z={Z_MID})\nRed tint = ground-truth ink", fontsize=11)

for i, idx in enumerate(chosen):
    cy = valid_ys[idx] + PATCH // 2
    cx = valid_xs[idx] + PATCH // 2
    patch = normalize(sl[cy-PATCH//2:cy+PATCH//2, cx-PATCH//2:cx+PATCH//2])
    lbl_p = labels_bin[cy-PATCH//2:cy+PATCH//2, cx-PATCH//2:cx+PATCH//2].astype(float)

    rgb = np.stack([patch, patch, patch], axis=-1)
    rgb[:, :, 0] = np.clip(rgb[:, :, 0] + lbl_p * 0.55, 0, 1)
    rgb[:, :, 1] = np.clip(rgb[:, :, 1] - lbl_p * 0.2,  0, 1)
    rgb[:, :, 2] = np.clip(rgb[:, :, 2] - lbl_p * 0.2,  0, 1)

    ink_frac = lbl_p.mean() * 100
    ax = axes[i // 4, i % 4]
    ax.imshow(np.clip(rgb, 0, 1))
    ax.set_title(f"({cx},{cy})  ink={ink_frac:.0f}%", fontsize=7)
    ax.axis("off")

plt.tight_layout(); plt.show()